In [ ]:
from pathlib import Path
import pandas as pd
import re

# 根目录
root = Path(r"G:\wxy_codes\Fetal_hydronephrosis\1第一批数据\batch1data_3")

# 如果存在 raw 文件夹，则自动进入 raw
data_root = root / "raw" if (root / "raw").exists() else root

# 需要统计的图像格式
target_exts = {".jpg", ".jpeg", ".bmp"}

# 匹配检查文件夹，例如 202412300188、2022081001116
visit_pattern = re.compile(r"^\d{8}")

def normalize_stem(stem: str) -> str:
    """
    去掉常见复制后缀，方便之后和 png 文件匹配
    例如：
    50_50_复制 -> 50_50
    50_50 - 副本 -> 50_50
    50_50 copy -> 50_50
    """
    s = stem.strip()
    s = re.sub(r"[_\-\s]*复制.*$", "", s)
    s = re.sub(r"[_\-\s]*副本.*$", "", s)
    s = re.sub(r"[_\-\s]*copy.*$", "", s, flags=re.IGNORECASE)
    s = re.sub(r"\s*\(\d+\)$", "", s)
    return s.strip()


def parse_image_location(img_path: Path):
    """
    从图片路径中解析 case_id、phase、visit_id、side、folder_type 等信息
    """
    rel_parts = img_path.relative_to(data_root).parts

    case_id = ""
    phase = ""
    visit_id = ""
    visit_index = None

    # 解析 case_id、phase、visit_id
    for i, part in enumerate(rel_parts):
        if part.startswith("case_"):
            case_id = part

            # case 后面一般是阶段文件夹：1prenatal / 2postnatal / 3postoperative
            if i + 1 < len(rel_parts):
                next_part = rel_parts[i + 1]
                if not visit_pattern.match(next_part):
                    phase = next_part

        if visit_pattern.match(part):
            visit_id = part
            visit_index = i
            break

    # 默认值
    side = "unknown"
    folder_type = "unknown"
    image_group = "unknown"

    if visit_index is not None:
        # 检查文件夹之后、图片文件之前的路径层级
        # 例如：
        # images
        # left/images
        # right/images
        # with_data_images
        # without_data_images
        sub_parts = rel_parts[visit_index + 1:-1]
        sub_parts_lower = [p.lower() for p in sub_parts]

        if "left" in sub_parts_lower:
            side = "left"
        elif "right" in sub_parts_lower:
            side = "right"
        else:
            side = "direct"

        if "with_data_images" in sub_parts_lower:
            folder_type = "with_data_images"
            image_group = "with_data"
        elif "without_data_images" in sub_parts_lower:
            folder_type = "without_data_images"
            image_group = "without_data"
        elif sub_parts_lower == ["images"]:
            folder_type = "direct_images"
            image_group = "images"
        elif sub_parts_lower == ["left", "images"]:
            folder_type = "left_images"
            image_group = "images"
        elif sub_parts_lower == ["right", "images"]:
            folder_type = "right_images"
            image_group = "images"
        else:
            folder_type = "/".join(sub_parts)
            image_group = "other"

    return case_id, phase, visit_id, side, folder_type, image_group


records = []

for img_path in data_root.rglob("*"):
    if not img_path.is_file():
        continue

    if img_path.suffix.lower() not in target_exts:
        continue

    case_id, phase, visit_id, side, folder_type, image_group = parse_image_location(img_path)

    records.append({
        "case_id": case_id,
        "phase": phase,
        "visit_id": visit_id,
        "side": side,
        "folder_type": folder_type,
        "image_group": image_group,
        "file_name": img_path.name,
        "file_stem": img_path.stem,
        "normalized_stem": normalize_stem(img_path.stem),
        "suffix": img_path.suffix.lower(),
        "image_folder": str(img_path.parent),
        "relative_path": str(img_path.relative_to(data_root)),
        "full_path": str(img_path)
    })

df = pd.DataFrame(records)

# 排序，方便检查
df = df.sort_values(
    by=["case_id", "phase", "visit_id", "folder_type", "side", "file_name"],
    ignore_index=True
)

# 保存为 Excel
save_path = root / "jpg_bmp_paths_with_case_visit_folder_type.xlsx"
df.to_excel(save_path, index=False)

print(f"共找到 {len(df)} 个 jpg/jpeg/bmp 文件")
print(f"表格已保存到：{save_path}")

df.head(30)

In [ ]:
from pathlib import Path
import pandas as pd
import shutil

# ========== 1. 路径配置 ==========

# 之前保存的 xlsx 表格
xlsx_path = Path(r"G:\wxy_codes\Fetal_hydronephrosis\1第一批数据\batch1data_3\jpg_bmp_paths_with_case_visit_folder_type.xlsx")

# 新处理好的 PNG 图像所在根目录
new_png_root = Path(r"G:\wxy_codes\Fetal_hydronephrosis\1第一批数据\batch1data_2\raw")

# 是否只预览，不真正删除和复制
dry_run = False

# 如果目标位置已经存在同名 png，是否覆盖
overwrite = True


# ========== 2. 读取 Excel ==========

df = pd.read_excel(xlsx_path)

print("表格列名：")
print(df.columns.tolist())
print(f"表格中共有 {len(df)} 条记录")


# ========== 3. 检查必要字段 ==========

required_cols = ["case_id", "visit_id", "file_stem", "full_path"]

missing_cols = [c for c in required_cols if c not in df.columns]

if missing_cols:
    raise ValueError(f"Excel 表格缺少必要列：{missing_cols}")


# ========== 4. 替换 jpg / bmp 为同名 png ==========

logs = []

for idx, row in df.iterrows():
    case_id = str(row["case_id"]).strip()
    visit_id = str(row["visit_id"]).strip()
    file_stem = str(row["file_stem"]).strip()

    old_img_path = Path(str(row["full_path"]))

    # 只处理 jpg/jpeg/bmp
    if old_img_path.suffix.lower() not in {".jpg", ".jpeg", ".bmp"}:
        logs.append({
            "status": "skip_not_jpg_bmp",
            "case_id": case_id,
            "visit_id": visit_id,
            "old_path": str(old_img_path),
            "new_png_path": "",
            "target_path": ""
        })
        continue

    # 严格同名匹配：50_50.jpg -> 50_50.png
    new_png_path = new_png_root / case_id / visit_id / "images" / f"{file_stem}.png"

    # 复制回旧图像原来的位置，只把后缀改为 .png
    target_png_path = old_img_path.with_suffix(".png")

    if not new_png_path.exists():
        logs.append({
            "status": "skip_new_png_not_found",
            "case_id": case_id,
            "visit_id": visit_id,
            "old_path": str(old_img_path),
            "new_png_path": str(new_png_path),
            "target_path": str(target_png_path)
        })
        continue

    if not old_img_path.exists():
        logs.append({
            "status": "skip_old_image_not_found",
            "case_id": case_id,
            "visit_id": visit_id,
            "old_path": str(old_img_path),
            "new_png_path": str(new_png_path),
            "target_path": str(target_png_path)
        })
        continue

    if target_png_path.exists() and not overwrite:
        logs.append({
            "status": "skip_target_png_exists",
            "case_id": case_id,
            "visit_id": visit_id,
            "old_path": str(old_img_path),
            "new_png_path": str(new_png_path),
            "target_path": str(target_png_path)
        })
        continue

    print(f"[替换] {old_img_path}")
    print(f"  来源 PNG：{new_png_path}")
    print(f"  目标 PNG：{target_png_path}")

    if not dry_run:
        # 先复制 png
        shutil.copy2(new_png_path, target_png_path)

        # 确认复制成功后，再删除原 jpg/bmp
        if target_png_path.exists():
            old_img_path.unlink()

    logs.append({
        "status": "replace",
        "case_id": case_id,
        "visit_id": visit_id,
        "old_path": str(old_img_path),
        "new_png_path": str(new_png_path),
        "target_path": str(target_png_path)
    })


# ========== 5. 保存日志 ==========

df_log = pd.DataFrame(logs)

log_path = xlsx_path.parent / "replace_jpg_bmp_with_png_log.xlsx"
df_log.to_excel(log_path, index=False)

print("\n处理完成")
print(df_log["status"].value_counts())
print(f"日志已保存到：{log_path}")

if dry_run:
    print("\n当前是预览模式，没有真正复制或删除。确认日志无误后，把 dry_run = False 再运行一次。")
else:
    print("\n已完成真实替换。")